# vLLM Colab Server v2

Google Colab GPU'sunda vLLM ile model serve et, Cloudflare Tunnel ile `api.ersamely.com` üzerinden eriş.

## Colab Secrets (gerekli)
| Secret | Açıklama |
|--------|----------|
| `HF_TOKEN` | HuggingFace erişim token'ı |
| `CF_TUNNEL_TOKEN` | Cloudflare named tunnel token |
| `VLLM_API_KEY` | API erişim anahtarı |

## Kullanım
1. **A**: Kurulum (bir kere)
2. **B**: Model seç + başlat
3. **C**: Tunnel bağla
4. **Model değiştir**: B'de `ACTIVE_MODEL` değiştir, B bölümünü tekrar çalıştır. Tunnel ayakta kalır.

---
# A) Kurulum (bir kere)

In [ ]:
import os
import subprocess

# \u2500\u2500\u2500 Environment \u2500\u2500\u2500
os.environ['VLLM_USE_FLASHINFER'] = '0'
os.environ['FLASHINFER_ENABLE_SM120'] = '0'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# \u2500\u2500\u2500 HuggingFace login \u2500\u2500\u2500
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    subprocess.run(
        ['huggingface-cli', 'login', '--token', hf_token, '--add-to-git-credential'],
        capture_output=True, text=True, check=True
    )
    print('\u2705 HuggingFace login')
except KeyError:
    print('\u26a0\ufe0f HF_TOKEN tan\u0131ml\u0131 de\u011fil \u2014 Secrets\'e ekle')
except Exception as e:
    print(f'\u26a0\ufe0f HF login atland\u0131: {e}')

# \u2500\u2500\u2500 GPU tespit \u2500\u2500\u2500
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamad\u0131! Runtime > Change runtime type > GPU se\u00e7')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_sm = torch.cuda.get_device_capability(0)  # (major, minor)
IS_BLACKWELL = gpu_sm[0] >= 10  # SM100+ = Blackwell
print(f'\u2705 GPU: {gpu_name} ({gpu_mem_gb:.1f} GB, SM{gpu_sm[0]}{gpu_sm[1]})')
print(f'   Blackwell: {"Evet" if IS_BLACKWELL else "Hay\u0131r"}')

# \u2500\u2500\u2500 Paketler \u2500\u2500\u2500
if IS_BLACKWELL:
    # Blackwell: T\u00dcM stack CUDA 13 olmal\u0131 (PyTorch + vLLM + FlashInfer)
    print('\u23f3 Blackwell \u2192 CUDA 13 stack y\u00fckleniyor...')
    # 1) libcudart.so.13 sa\u011fla (driver 13 destekliyor ama runtime yok)
    import glob, subprocess as _sp, pathlib
    _sp.run(['apt-get', 'update', '-qq'], capture_output=True)
    _sp.run(['apt-get', 'install', '-y', '-qq', 'cuda-cudart-13-0'], capture_output=True)
    # Symlink fallback
    _found13 = glob.glob('/usr/local/cuda*/lib64/libcudart.so.13*')
    if not _found13:
        _existing = glob.glob('/usr/local/cuda/lib64/libcudart.so*')
        if _existing:
            _t = pathlib.Path('/usr/local/cuda/lib64/libcudart.so.13')
            if not _t.exists():
                _t.symlink_to(sorted(_existing)[-1])
    for p in glob.glob('/usr/local/cuda*/lib64'):
        os.environ['LD_LIBRARY_PATH'] = p + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    # 2) uv ile CUDA 13 stack kur (PyTorch cu130 + vLLM cu130 + FlashInfer)
    !pip -q install uv 2>&1 | tail -1
    # PyTorch'u CUDA 13 olarak ZORLA yeniden kur (Colab default'u 12.8)
    !uv pip install torch torchvision --torch-backend cu130 --system --reinstall-package torch --reinstall-package torchvision -q 2>&1 | tail -3
    !pip uninstall -y torchaudio 2>/dev/null
    !uv pip install vllm httpx huggingface_hub --torch-backend cu130 --system -q 2>&1 | tail -5
    # 3) CUDA 13 nvcc (FlashInfer JIT i\u00e7in \u015fart)
    # NVIDIA apt repo ekle ve cuda toolkit 13 kur
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb 2>&1 | tail -1
    !dpkg -i /tmp/cuda-keyring.deb > /dev/null 2>&1
    !apt-get update -qq 2>&1 | tail -1
    !apt-get install -y -qq cuda-nvcc-13-0 cuda-cudart-dev-13-0 2>&1 | tail -3
    import glob as _g
    # CUDA 13 path bul
    _cuda13_homes = _g.glob('/usr/local/cuda-13*')
    if _cuda13_homes:
        _cuda13 = sorted(_cuda13_homes)[-1]
        os.environ['CUDA_HOME'] = _cuda13
        os.environ['PATH'] = _cuda13 + '/bin:' + os.environ.get('PATH', '')
        print(f'\u2705 nvcc: {_cuda13}/bin/nvcc')
    else:
        # Fallback: mevcut sistem nvcc kullan
        os.environ['CUDA_HOME'] = '/usr/local/cuda'
        print('\u26a0\ufe0f CUDA 13 toolkit kurulamad\u0131, /usr/local/cuda kullan\u0131l\u0131yor')
    print('\u2705 CUDA 13 stack kuruldu')
else:
    # Non-Blackwell (A100/L4): vLLM 0.9.x (CUDA 12.8 native, libcudart.so.13 bug yok)
    _cuda_ver = torch.version.cuda
    print(f'\u23f3 CUDA {_cuda_ver} \u2192 vLLM 0.9.x y\u00fckleniyor...')
    !pip -q install "vllm>=0.9,<0.10" httpx huggingface_hub 2>&1 | tail -5
    !pip uninstall flashinfer flashinfer-python -y 2>/dev/null

try:
    import importlib, torch as _t
    importlib.reload(_t)
    print(f'\u2705 torch.version.cuda = {_t.version.cuda}')
except Exception:
    print('\u26a0\ufe0f torch reload ba\u015far\u0131s\u0131z \u2014 Runtime restart gerekli')
import vllm
print(f'\u2705 vLLM {vllm.__version__}')

# \u2500\u2500\u2500 LD_LIBRARY_PATH \u2500\u2500\u2500
import glob
cuda_paths = [
    '/usr/local/cuda/lib64',
    '/usr/lib/x86_64-linux-gnu',
]
cuda_paths += glob.glob('/usr/local/lib/python*/dist-packages/nvidia/*/lib')
for p in cuda_paths:
    if os.path.exists(p):
        os.environ['LD_LIBRARY_PATH'] = p + ':' + os.environ.get('LD_LIBRARY_PATH', '')

# \u2500\u2500\u2500 Cloudflared \u2500\u2500\u2500
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

# \u2500\u2500\u2500 Disk alan\u0131 kontrol \u2500\u2500\u2500
import shutil
disk = shutil.disk_usage('/')
free_gb = disk.free / 1024**3
print(f'\n\u2705 Disk: {free_gb:.1f} GB bo\u015f')
if free_gb < 20:
    print('\u26a0\ufe0f Disk alan\u0131 d\u00fc\u015f\u00fck! B\u00fcy\u00fck modeller i\u00e7in sorun olabilir.')

LOG_DIR = '/content/logs'
os.makedirs(LOG_DIR, exist_ok=True)

!nvidia-smi | head -12
print('\n\u2705 Kurulum tamam')

---
# B) Model Se\u00e7 ve Ba\u015flat

`ACTIVE_MODEL` de\u011fi\u015ftirip bu b\u00f6l\u00fcm\u00fc tekrar \u00e7al\u0131\u015ft\u0131rman yeterli.  
Tunnel (C b\u00f6l\u00fcm\u00fc) ayakta kal\u0131r, kesinti olmaz.

In [ ]:
from google.colab import userdata

# \u2554\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2557
# \u2551  MODEL KATALO\u011eU                                                \u2551
# \u2551  vram_gb: Tahmini minimum VRAM gereksinimi                       \u2551
# \u255a\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u255d
MODELS = {
    'hy-mt2-30b-a3b-fp8': {
        'repo': 'tencent/Hy-MT2-30B-A3B-FP8',
        'max_model_len': 65536,
        'vram_gb': 22,
        'fp8': True,
        'extra_args': [],
    },
    'hy-mt2-7b-fp8': {
        'repo': 'tencent/Hy-MT2-7B-FP8',
        'max_model_len': 65536,
        'vram_gb': 10,
        'fp8': True,
        'extra_args': [],
    },
    'qwen3.6-35b-a3b': {
        'repo': 'Qwen/Qwen3.6-35B-A3B',
        'max_model_len': 65536,
        'vram_gb': 24,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
    'qwen3.6-35b-a3b-fp8': {
        'repo': 'Qwen/Qwen3.6-35B-A3B-FP8',
        'max_model_len': 65536,
        'vram_gb': 22,
        'fp8': True,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
    'qwen3.5-9b-fp8': {
        'repo': 'Hyper-AI/Qwen3.5-9B-fp8',
        'max_model_len': 65536,
        'vram_gb': 12,
        'fp8': True,
        'extra_args': [],
    },
    'qwen3.5-9b-awq': {
        'repo': 'cyankiwi/Qwen3.5-9B-AWQ-4bit',
        'max_model_len': 65536,
        'vram_gb': 8,
        'extra_args': ['--quantization', 'awq'],
    },
    'qwen3-14b-fp8': {
        'repo': 'nvidia/Qwen3-14B-FP8',
        'max_model_len': 65536,
        'vram_gb': 16,
        'fp8': True,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
    'qwen3-14b-awq': {
        'repo': 'Qwen/Qwen3-14B-AWQ',
        'max_model_len': 65536,
        'vram_gb': 10,
        'extra_args': ['--quantization', 'awq', '--reasoning-parser', 'qwen3'],
    },
    'qwen3-8b-fp8': {
        'repo': 'Qwen/Qwen3-8B-FP8',
        'max_model_len': 65536,
        'vram_gb': 10,
        'fp8': True,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
}

# \u2554\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2557
# \u2551  AKT\u0130F MODEL \u2014 Bunu de\u011fi\u015ftir, B b\u00f6l\u00fcm\u00fcn\u00fc tekrar \u00e7al\u0131\u015ft\u0131r      \u2551
# \u255a\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u255d
ACTIVE_MODEL = 'hy-mt2-30b-a3b-fp8'

# \u2500\u2500\u2500 Resolve \u2500\u2500\u2500
if ACTIVE_MODEL not in MODELS:
    print('Mevcut modeller:')
    for name, m in MODELS.items():
        print(f'  - {name:30s} ({m["repo"]}) ~{m.get("vram_gb", "?")} GB')
    raise ValueError(f'Model bulunamad\u0131: {ACTIVE_MODEL}')

_m = MODELS[ACTIVE_MODEL]
MODEL_REPO    = _m['repo']
MAX_MODEL_LEN = _m.get('max_model_len', 32768)
GPU_MEM_UTIL  = _m.get('gpu_mem_util', 0.98)
EXTRA_ARGS    = _m.get('extra_args', [])
VRAM_REQ      = _m.get('vram_gb', 0)
VLLM_PORT     = 8090

API_KEY       = userdata.get('VLLM_API_KEY')
VLLM_BASE_URL = f'http://localhost:{VLLM_PORT}'
TUNNEL_URL    = 'https://api.ersamely.com'

# \u2500\u2500\u2500 VRAM uyar\u0131s\u0131 \u2500\u2500\u2500
import torch
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_sm = torch.cuda.get_device_capability(0)
_is_blackwell = gpu_sm[0] >= 10

if VRAM_REQ and gpu_mem_gb < VRAM_REQ:
    print(f'\u26a0\ufe0f UYARI: {ACTIVE_MODEL} ~{VRAM_REQ} GB VRAM gerektirir, GPU\'da {gpu_mem_gb:.0f} GB var!')
    print(f'   Daha k\u00fc\u00e7\u00fck bir model se\u00e7 veya max_model_len d\u00fc\u015f\u00fcr.\n')

if _m.get('fp8') and not _is_blackwell:
    print(f'\u26a0\ufe0f UYARI: {ACTIVE_MODEL} FP8 model \u2014 vLLM 0.8.x kullan\u0131l\u0131yor (flashinfer FP8, SM100+ gerektirir).')
    print(f'   Sorun ya\u015farsan AWQ modelleri dene: qwen3.5-9b-awq, qwen3-14b-awq\n')

# \u2500\u2500\u2500 Uyumlu modeller \u2500\u2500\u2500
compat = [
    n for n, m in MODELS.items()
    if m.get('vram_gb', 999) <= gpu_mem_gb
    and (not m.get('fp8') or _is_blackwell)
]

print(f'\u250c{"\u2500"*58}\u2510')
print(f'\u2502 {ACTIVE_MODEL:56s} \u2502')
print(f'\u251c{"\u2500"*58}\u2524')
print(f'\u2502  repo:       {MODEL_REPO[:42]:42s} \u2502')
print(f'\u2502  max_len:    {MAX_MODEL_LEN:<42} \u2502')
print(f'\u2502  gpu_util:   {GPU_MEM_UTIL:<42} \u2502')
print(f'\u2502  vram_req:   ~{VRAM_REQ} GB{" ":36s} \u2502')
print(f'\u2502  extra:      {str(EXTRA_ARGS)[:42]:42s} \u2502')
print(f'\u2514{"\u2500"*58}\u2518')
print(f'\nUyumlu modeller ({gpu_mem_gb:.0f} GB GPU): {" | ".join(compat)}')

In [ ]:
import os
import subprocess
import time

import requests
import torch

LOG_PATH = f'{LOG_DIR}/vllm.log'

# \u2500\u2500\u2500 \u00d6nceki vLLM'i durdur \u2500\u2500\u2500
print(f'\U0001f6d1 \u00d6nceki vLLM durduruluyor...')
subprocess.run(['pkill', '-f', 'vllm serve'], capture_output=True)
time.sleep(3)
torch.cuda.empty_cache()

# \u2500\u2500\u2500 vLLM ba\u015flat \u2500\u2500\u2500
gpu_sm = torch.cuda.get_device_capability(0)
_is_bw = gpu_sm[0] >= 10
env = {**os.environ}
if not _is_bw:
    env['FLASHINFER_ENABLE_SM120'] = '0'
cmd = [
    'vllm', 'serve', MODEL_REPO,
    '--host', '0.0.0.0',
    '--port', str(VLLM_PORT),
    '--max-model-len', str(MAX_MODEL_LEN),
    '--gpu-memory-utilization', str(GPU_MEM_UTIL),
    '--trust-remote-code',
    '--api-key', API_KEY,
    '--enforce-eager',
    *EXTRA_ARGS,
]

with open(LOG_PATH, 'w') as f:
    proc = subprocess.Popen(
        cmd, stdout=f, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, env=env
    )

print(f'\U0001f680 {ACTIVE_MODEL} ba\u015flat\u0131ld\u0131 (PID: {proc.pid})')
print(f'   cmd: vllm serve {MODEL_REPO} ...')


def get_log_tail(n=5):
    """Log dosyas\u0131n\u0131n son n sat\u0131r\u0131n\u0131 oku."""
    try:
        with open(LOG_PATH) as f:
            return ''.join(f.readlines()[-n:])
    except OSError:
        return ''


def detect_phase(log):
    """Log i\u00e7eri\u011fine g\u00f6re mevcut a\u015famay\u0131 tespit et."""
    lower = log.lower()
    if ('error' in lower or 'traceback' in lower) and 'retrying' not in lower:
        return '\u274c HATA'
    if 'downloading' in lower or 'fetching' in lower:
        return '\U0001f4e5 \u0130ndiriliyor'
    if 'loading model' in lower or 'loading weights' in lower:
        return '\U0001f504 Model y\u00fckleniyor'
    if 'compiling' in lower or 'graph capture' in lower:
        return '\U0001f527 Derleniyor'
    if 'warming up' in lower or 'cuda graph' in lower:
        return '\U0001f525 CUDA warmup'
    if 'tokenizer' in lower and 'loading' in lower:
        return '\U0001f4dd Tokenizer y\u00fckleniyor'
    if 'started server' in lower or 'uvicorn running' in lower:
        return '\u2705 Haz\u0131r'
    return '\u23f3 Ba\u015flat\u0131l\u0131yor'


# \u2500\u2500\u2500 Bekle \u2500\u2500\u2500
TIMEOUT = 900  # 15 dakika (b\u00fcy\u00fck modeller i\u00e7in)
t0 = time.time()
ready = False
last_phase = ''

while time.time() - t0 < TIMEOUT:
    elapsed = int(time.time() - t0)
    phase = detect_phase(get_log_tail())

    # Health check
    try:
        if requests.get(f'{VLLM_BASE_URL}/health', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass

    # Process crash
    if proc.poll() is not None:
        print(f'\u274c vLLM \u00e7\u00f6kt\u00fc! (exit: {proc.returncode})')
        print('\u2500' * 60)
        print(get_log_tail(50))
        break

    # Status (sadece de\u011fi\u015fen fazlar\u0131 g\u00f6ster)
    if phase != last_phase:
        print(f'[{elapsed:3d}s] {phase}')
        last_phase = phase
    elif elapsed % 30 == 0:  # Her 30s'de bir heartbeat
        print(f'[{elapsed:3d}s] {phase}')

    if '\u274c' in phase:
        print('\u2500' * 60)
        print(get_log_tail(20))
        break

    time.sleep(5)

# \u2500\u2500\u2500 Sonu\u00e7 \u2500\u2500\u2500
if ready:
    elapsed = int(time.time() - t0)
    print(f'\n\u2705 {ACTIVE_MODEL} haz\u0131r! ({elapsed}s)')
    # Smoke test
    try:
        r = requests.post(
            f'{VLLM_BASE_URL}/v1/chat/completions',
            headers={'Authorization': f'Bearer {API_KEY}'},
            json={
                'model': MODEL_REPO,
                'messages': [{'role': 'user', 'content': 'Say hello in one sentence.'}],
                'max_tokens': 64,
            },
            timeout=120,
        )
        if r.status_code == 200:
            data = r.json()
            msg = data['choices'][0]['message']
            txt = msg.get('content') or msg.get('reasoning_content') or 'No content'
            usage = data.get('usage', {})
            print(f'   \U0001f4ac {txt[:200]}')
            if usage:
                print(f'   \U0001f4ca tokens: {usage.get("prompt_tokens", 0)} in / {usage.get("completion_tokens", 0)} out')
        else:
            print(f'   \u26a0\ufe0f HTTP {r.status_code}: {r.text[:200]}')
    except requests.RequestException as e:
        print(f'   \u26a0\ufe0f Smoke test ba\u015far\u0131s\u0131z: {e}')
else:
    if proc.poll() is None:
        print(f'\n\u274c Timeout ({TIMEOUT}s)! vLLM hala \u00e7al\u0131\u015f\u0131yor ama health check ge\u00e7miyor.')
    print('\u2500' * 60)
    print(get_log_tail(50))

---
# C) Cloudflare Tunnel

Bir kere \u00e7al\u0131\u015ft\u0131r. Model de\u011fi\u015fti\u011finde tunnel ayakta kal\u0131r.

`api.ersamely.com` \u2192 `localhost:8090`

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata

subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

token = userdata.get('CF_TUNNEL_TOKEN')

CF_LOG = f'{LOG_DIR}/cloudflared.log'
with open(CF_LOG, 'w') as f:
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
        stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
    )

print(f'\U0001f310 Tunnel ba\u015flat\u0131ld\u0131 (PID: {cf_proc.pid})')
time.sleep(5)

# Tunnel kontrol (retry ile)
tunnel_ok = False
for attempt in range(3):
    try:
        r = requests.get(
            f'{TUNNEL_URL}/health',
            headers={'Authorization': f'Bearer {API_KEY}'},
            timeout=10,
        )
        if r.status_code == 200:
            tunnel_ok = True
            break
    except requests.RequestException:
        time.sleep(3)

if tunnel_ok:
    print(f'\u2705 Tunnel aktif: {TUNNEL_URL}')
    print(f'\n   Open WebUI ba\u011flant\u0131s\u0131:')
    print(f'   URL: {TUNNEL_URL}/v1')
    print(f'   API Key: (Colab Secrets > VLLM_API_KEY)')
elif cf_proc.poll() is None:
    print(f'\u2705 Tunnel \u00e7al\u0131\u015f\u0131yor: {TUNNEL_URL}')
    print('   (vLLM haz\u0131r olunca eri\u015filebilir olacak)')
else:
    print('\u274c Tunnel ba\u015far\u0131s\u0131z!')
    try:
        with open(CF_LOG) as f:
            print(f.read()[-500:])
    except OSError:
        pass

In [ ]:
import time
from datetime import datetime, timezone

import requests


def get_gpu_stats():
    """GPU memory ve utilization bilgisi."""
    try:
        import subprocess
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=5
        )
        if out.returncode == 0:
            parts = out.stdout.strip().split(', ')
            mem_used, mem_total, gpu_util = int(parts[0]), int(parts[1]), int(parts[2])
            return f'{mem_used}/{mem_total} MB ({gpu_util}%)'
    except Exception:
        pass
    return '?'


print(f'Canl\u0131 tutma: {TUNNEL_URL} | Model: {ACTIVE_MODEL}')
print('Durdurmak i\u00e7in interrupt et (\u25a0 butonu).\n')

start_time = time.time()
check_count = 0

while True:
    try:
        local_ok = requests.get(f'{VLLM_BASE_URL}/health', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    try:
        tunnel_ok = requests.get(
            f'{TUNNEL_URL}/health',
            headers={'Authorization': f'Bearer {API_KEY}'},
            timeout=10
        ).status_code == 200
    except requests.RequestException:
        tunnel_ok = False

    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    uptime_min = int((time.time() - start_time) / 60)
    l = '\u2705' if local_ok else '\u274c'
    t = '\u2705' if tunnel_ok else '\u274c'
    gpu = get_gpu_stats()

    print(f'{now} | {ACTIVE_MODEL} | vLLM: {l} | Tunnel: {t} | GPU: {gpu} | up: {uptime_min}m')

    check_count += 1
    time.sleep(30)

---
# D) Diagnostik (opsiyonel)

In [ ]:
import requests

print('=== vLLM Log (son 30 sat\u0131r) ===')
try:
    with open(f'{LOG_DIR}/vllm.log') as f:
        lines = f.readlines()
        print(''.join(lines[-30:]))
except FileNotFoundError:
    print('Log dosyas\u0131 yok')

print('\n=== Model Bilgisi ===')
try:
    r = requests.get(
        f'{VLLM_BASE_URL}/v1/models',
        headers={'Authorization': f'Bearer {API_KEY}'},
        timeout=5
    )
    if r.status_code == 200:
        models = r.json().get('data', [])
        for m in models:
            print(f'  - {m["id"]}')
    else:
        print(f'  HTTP {r.status_code}')
except requests.RequestException as e:
    print(f'  Ba\u011flant\u0131 yok: {e}')

print('\n=== GPU Durumu ===')
!nvidia-smi

print('\n=== Disk ===')
!df -h / | tail -1